# Prepare Dynamic Wflow Handoff

Collect or reuse the shared Wflow warmup baseline, then prepare and QA the Wflow discharge time series consumed by SFINCS for one Event-Catalog event.

Stage Contract
- Requires: built Wflow submodels with SFINCS handoff gauges, AORC source access, and reviewed NHDPlus/STREAM-geo river geometry.
- Produces: shared Wflow warmup forcing, Wflow event forcing, dynamic Wflow `sfincs_discharge.nc`, dynamic handoff QA, and acceptance JSON.
- Next: run `c_run_example.ipynb` after the handoff status is accepted.


In [ ]:
import os
import warnings
from pathlib import Path

os.environ.pop("DEBUG", None)
os.environ.setdefault("MPLCONFIGDIR", "/tmp/flood-rm-matplotlib")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore")

import pandas as pd
from IPython.display import display

from design_events.collect_sources import collect_aorc_wflow_baseline_warmup
from sfincs_runs.config import load_runtime
from sfincs_runs.scenarios import plan_inland_coupled_example
from wflow_runs import (
    build_event_meteo_forcing,
    plan_dynamic_wflow_handoff,
    plan_wflow_streamflow_realization,
    plan_wflow_baseline_warmup_state,
    prepare_dynamic_wflow_handoff,
    validate_river_geometry_readiness,
    validate_warmup_forcing,
    validate_wflow_instates,
)
from wflow_runs.build_plan import validate_wflow_staticmaps_physics

notebook_root = Path.cwd().resolve()
location_root = notebook_root.parents[1]
repo_root = location_root.parents[1]
config, paths = load_runtime(location_root / "config.yaml")
config["wflow"]["domain_set"]["review_required"] = False
print(f"location: {location_root.name}  |  root: {location_root}")


## 1 - Select Event and Plan Handoff


In [ ]:
EVENT_ID = None  # None -> same default event as c_run_example; or e.g. "design_0369"
CATALOG_PATH = "data/event_catalog/catalog/probability_catalog.csv"

example_plan = plan_inland_coupled_example(
    config,
    {"location_root": location_root},
    catalog_path=CATALOG_PATH,
    event_id=EVENT_ID,
)
if example_plan.event_id is None:
    raise RuntimeError("Coupled example event could not be selected; resolve upstream plan issues before dynamic handoff prep.")
if example_plan.issues:
    print("Planning issues to review:")
    for issue in example_plan.issues:
        print("  -", issue)

event_id = str(example_plan.event_id)
catalog = pd.read_csv(location_root / CATALOG_PATH)
catalog["event_id"] = catalog["event_id"].astype(str)
selected = catalog[catalog["event_id"] == event_id].iloc[0]

handoff_plan = plan_dynamic_wflow_handoff(config, location_root, event_id, catalog_path=location_root / CATALOG_PATH)
display(handoff_plan)

streamflow_realization = plan_wflow_streamflow_realization(config, location_root, event_id, catalog_path=location_root / CATALOG_PATH)
display(streamflow_realization)
if streamflow_realization["status"].eq("failed").any():
    raise RuntimeError("Selected USGS/POT streamflow realization is not ready for Wflow external river inflow.")


## 2 - Verify Wflow Source Geometry and Static Maps


In [ ]:
river_geometry = location_root / config["collection"]["national_hydrography"]["river_geometry"]
display(validate_river_geometry_readiness(river_geometry, raise_on_error=False))

base_root = location_root / config["wflow"].get("base_model_root", "data/wflow/base")
qa_rows = []
for submodel in config["wflow"]["domain_set"].get("submodels", []):
    model_root = base_root / str(submodel["wflow_submodel_id"])
    if (model_root / "staticmaps.nc").exists():
        report = validate_wflow_staticmaps_physics(model_root, river_upa_km2=config["inland_coupling"]["discharge_forcing"].get("river_upa_km2"), raise_on_error=False)
        report.insert(0, "submodel_id", str(submodel["wflow_submodel_id"]))
        qa_rows.append(report)
staticmap_qa = pd.concat(qa_rows, ignore_index=True) if qa_rows else pd.DataFrame()
display(staticmap_qa)
if not staticmap_qa.empty and staticmap_qa["status"].isin(["failed", "review_required"]).any():
    raise RuntimeError("Wflow staticmap QA must pass before dynamic SFINCS handoff.")


## 3 - Collect Shared Warmup AORC Forcing


In [ ]:
baseline_plan = plan_wflow_baseline_warmup_state(config, location_root)
display(baseline_plan)

collect_baseline_warmup = True
if collect_baseline_warmup:
    warmup_collection = collect_aorc_wflow_baseline_warmup(config, paths, force=False)
    display(pd.Series(warmup_collection, name="aorc_wflow_baseline_warmup"))

warmup_forcing = validate_warmup_forcing(
    config,
    location_root,
    event_id,
    reference_time=baseline_plan["baseline_reference_time"],
    raise_on_error=False,
)
display(warmup_forcing)
if warmup_forcing["status"].eq("failed").any():
    raise RuntimeError("Collect the shared AORC warmup baseline before running dynamic Wflow handoff.")

instate_readiness = validate_wflow_instates(config, location_root, raise_on_error=False)
display(instate_readiness)
if not instate_readiness.empty and instate_readiness["status"].eq("failed").any():
    raise RuntimeError("Create or promote native Wflow instate/instates.nc from the shared warmup baseline before event replay.")


## 4 - Stage Wflow Event Meteo


In [ ]:
meteo_report = build_event_meteo_forcing(
    config,
    location_root,
    event_id,
    catalog_path=location_root / CATALOG_PATH,
    overwrite=False,
)
display(pd.Series(meteo_report, name="wflow_event_meteo_forcing"))


## 5 - Run Dynamic Wflow Handoff QA


In [ ]:
run_dynamic_wflow = True

handoff_report = prepare_dynamic_wflow_handoff(
    config,
    location_root,
    event_id,
    catalog_path=location_root / CATALOG_PATH,
    execute=run_dynamic_wflow,
)
display(handoff_report)


## 6 - Acceptance Artifact


In [ ]:
acceptance_path = location_root / handoff_plan["dynamic_handoff_acceptance"]
if acceptance_path.exists():
    display(pd.read_json(acceptance_path, typ="series"))
else:
    print("Dynamic handoff planned but not executed yet:", acceptance_path)
